In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import f1_score, balanced_accuracy_score
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, SGDClassifier

In [2]:
train_df = pd.read_csv('train_data.csv')
test_df = pd.read_csv('test_data.csv')

In [3]:
DIM_NAMES = ["target_1", "target_2", "target_3", "target_4"]
TEXT_COL = "text"
LABEL_COLS = ["target_1", "target_2", "target_3", "target_4"]

MBTI_TYPES = [
    "INTP","INTJ","INFJ","INFP","ENTP","ENTJ","ENFJ","ENFP",
    "ISTP","ISTJ","ISFJ","ISFP","ESTP","ESTJ","ESFJ","ESFP"
]
pat_mbti = re.compile(r"\b(" + "|".join(MBTI_TYPES) + r")\b", re.IGNORECASE)

def compute_fulltype_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    correct = (y_true == y_pred).sum(axis=1)
    return {
        "Full MBTI Type Accuracy (All 4 correct)": float((correct == 4).mean()),
        "At least 3 correct": float((correct >= 3).mean()),
        "At least 2 correct": float((correct >= 2).mean()),
        "At least 1 correct": float((correct >= 1).mean()),
    }

def per_dim_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> pd.DataFrame:
    rows = []
    for k, name in enumerate(DIM_NAMES):
        rows.append({
            "dimension": name,
            "macro_f1": f1_score(y_true[:, k], y_pred[:, k], average="macro"),
            "balanced_acc": balanced_accuracy_score(y_true[:, k], y_pred[:, k]),
        })
    return pd.DataFrame(rows).set_index("dimension")

def tune_thresholds(probs_val: np.ndarray, y_val: np.ndarray) -> np.ndarray:
    thrs = np.zeros(y_val.shape[1], dtype=float)
    for k in range(y_val.shape[1]):
        best_t, best_f1 = 0.5, -1.0
        for t in np.linspace(0.05, 0.95, 91):
            pred = (probs_val[:, k] >= t).astype(int)
            f1 = f1_score(y_val[:, k], pred, average="macro")
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thrs[k] = best_t
    return thrs

In [4]:
y_train_full = train_df[LABEL_COLS].astype(int).values
y_test = test_df[LABEL_COLS].astype(int).values


In [5]:
tr_df, va_df, y_tr, y_va = train_test_split(
    train_df, y_train_full, test_size=0.2, random_state=42
)

len(tr_df), len(va_df), len(test_df)

(71850, 17963, 22454)

In [6]:
use_text = 'text'

In [7]:
word_vec = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.9,
    sublinear_tf=True,
    strip_accents="unicode",
)

char_vec = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    sublinear_tf=True,
)

tfidf = FeatureUnion([("word", word_vec), ("char", char_vec)])
tfidf.fit(tr_df[use_text])

X_tr = tfidf.transform(tr_df[use_text])
X_va = tfidf.transform(va_df[use_text])
X_te = tfidf.transform(test_df[use_text])

X_tr.shape, X_te.shape

((71850, 2300058), (22454, 2300058))

In [8]:
def train_predict_linearsvc(X_tr, y_tr, X_te):
    y_pred = np.zeros((X_te.shape[0], y_tr.shape[1]), dtype=int)
    for k in range(y_tr.shape[1]):
        clf = LinearSVC()
        clf.fit(X_tr, y_tr[:, k])
        y_pred[:, k] = clf.predict(X_te).astype(int)
    return y_pred

y_pred_svc = train_predict_linearsvc(X_tr, y_tr, X_te)

print("=== LinearSVC ===")
display(per_dim_metrics(y_test, y_pred_svc))
print(compute_fulltype_metrics(y_test, y_pred_svc))

=== LinearSVC ===


,macro_f1,balanced_acc
dimension,,
target_1,0.761338,0.744146
target_2,0.827423,0.821612
target_3,0.804107,0.802493
target_4,0.768930,0.761798


{'Full MBTI Type Accuracy (All 4 correct)': 0.4893114812505567, 'At least 3 correct': 0.821011846441614, 'At least 2 correct': 0.9629910038300525, 'At least 1 correct': 0.996793444375167}


In [9]:
def train_predict_proba_logreg(X_tr, y_tr, X):
    probs = np.zeros((X.shape[0], y_tr.shape[1]), dtype=float)
    for k in range(y_tr.shape[1]):
        clf = LogisticRegression(
            solver="saga",
            max_iter=300,
            n_jobs=-1,
            class_weight="balanced",
        )
        clf.fit(X_tr, y_tr[:, k])
        probs[:, k] = clf.predict_proba(X)[:, 1]
    return probs

probs_va_lr = train_predict_proba_logreg(X_tr, y_tr, X_va)
probs_te_lr = train_predict_proba_logreg(X_tr, y_tr, X_te)

thr_lr = tune_thresholds(probs_va_lr, y_va)

y_pred_lr_default = (probs_te_lr >= 0.5).astype(int)
y_pred_lr_tuned = (probs_te_lr >= thr_lr.reshape(1, -1)).astype(int)

print("=== LogReg balanced (thr=0.5) ===")
display(per_dim_metrics(y_test, y_pred_lr_default))
print(compute_fulltype_metrics(y_test, y_pred_lr_default))

print("\n=== LogReg balanced (tuned thresholds) ===")
print({LABEL_COLS[i]: float(thr_lr[i]) for i in range(4)})
display(per_dim_metrics(y_test, y_pred_lr_tuned))
print(compute_fulltype_metrics(y_test, y_pred_lr_tuned))

=== LogReg balanced (thr=0.5) ===


,macro_f1,balanced_acc
dimension,,
target_1,0.770894,0.778492
target_2,0.821879,0.827235
target_3,0.802537,0.803789
target_4,0.766343,0.770578


{'Full MBTI Type Accuracy (All 4 correct)': 0.47318963213681303, 'At least 3 correct': 0.807027701077759, 'At least 2 correct': 0.9571123185178587, 'At least 1 correct': 0.9947002761200677}

=== LogReg balanced (tuned thresholds) ===
{'target_1': 0.49999999999999994, 'target_2': 0.5199999999999999, 'target_3': 0.49999999999999994, 'target_4': 0.5099999999999999}


,macro_f1,balanced_acc
dimension,,
target_1,0.770894,0.778492
target_2,0.824570,0.826966
target_3,0.802537,0.803789
target_4,0.768131,0.770453


{'Full MBTI Type Accuracy (All 4 correct)': 0.4773314331522223, 'At least 3 correct': 0.8091208693328583, 'At least 2 correct': 0.9574686024761735, 'At least 1 correct': 0.9949674890888037}


In [10]:
def train_predict_proba_sgd(X_tr, y_tr, X):
    probs = np.zeros((X.shape[0], y_tr.shape[1]), dtype=float)
    for k in range(y_tr.shape[1]):
        clf = SGDClassifier(
            loss="log_loss",
            alpha=1e-5,
            max_iter=30,
            n_jobs=-1,
            class_weight="balanced",
            random_state=42,
        )
        clf.fit(X_tr, y_tr[:, k])
        probs[:, k] = clf.predict_proba(X)[:, 1]
    return probs

probs_va_sgd = train_predict_proba_sgd(X_tr, y_tr, X_va)
probs_te_sgd = train_predict_proba_sgd(X_tr, y_tr, X_te)

thr_sgd = tune_thresholds(probs_va_sgd, y_va)

y_pred_sgd_default = (probs_te_sgd >= 0.5).astype(int)
y_pred_sgd_tuned = (probs_te_sgd >= thr_sgd.reshape(1, -1)).astype(int)

print("=== SGD(log_loss) balanced (thr=0.5) ===")
display(per_dim_metrics(y_test, y_pred_sgd_default))
print(compute_fulltype_metrics(y_test, y_pred_sgd_default))

print("\n=== SGD(log_loss) balanced (tuned thresholds) ===")
print({LABEL_COLS[i]: float(thr_sgd[i]) for i in range(4)})
display(per_dim_metrics(y_test, y_pred_sgd_tuned))
print(compute_fulltype_metrics(y_test, y_pred_sgd_tuned))

=== SGD(log_loss) balanced (thr=0.5) ===


,macro_f1,balanced_acc
dimension,,
target_1,0.769242,0.782914
target_2,0.822272,0.829826
target_3,0.805253,0.805937
target_4,0.758729,0.775864


{'Full MBTI Type Accuracy (All 4 correct)': 0.4621893649238443, 'At least 3 correct': 0.8025741515988243, 'At least 2 correct': 0.9554199697158635, 'At least 1 correct': 0.9947893471096464}

=== SGD(log_loss) balanced (tuned thresholds) ===
{'target_1': 0.5399999999999999, 'target_2': 0.5399999999999999, 'target_3': 0.49999999999999994, 'target_4': 0.57}


,macro_f1,balanced_acc
dimension,,
target_1,0.772134,0.775672
target_2,0.826766,0.828707
target_3,0.805253,0.805937
target_4,0.769391,0.773214


{'Full MBTI Type Accuracy (All 4 correct)': 0.4817404471363677, 'At least 3 correct': 0.8126391734212167, 'At least 2 correct': 0.958047563908435, 'At least 1 correct': 0.9954128440366973}
